In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load
from copy import deepcopy
import json
import os

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
from sklearn.experimental import enable_hist_gradient_boosting
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.multioutput import MultiOutputRegressor
from sklearn.preprocessing import LabelEncoder

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory


#for dirname, _, filenames in os.walk('/kaggle/input'):
#    for filename in filenames:
#        print(os.path.join(dirname, filename))

# You can write up to 5GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session


/usr/local/lib/python3.11/dist-packages/sklearn/experimental/enable_hist_gradient_boosting.py:16: UserWarning: Since version 1.0, it is not needed to import enable_hist_gradient_boosting anymore. HistGradientBoostingClassifier and HistGradientBoostingRegressor are now stable and can be normally imported from sklearn.ensemble.
  warnings.warn(


In [2]:
data_train = [json.loads(line) for line in open('../input/stanford-covid-vaccine/train.json','r')]
data_test = [json.loads(linha) for linha in open('../input/stanford-covid-vaccine/test.json','r')]
test_set = pd.read_csv('../input/stanford-covid-vaccine/sample_submission.csv')


In [3]:
for jason in data_train:
    jason['step'] = list(range(jason['seq_scored']))
    jason['sequence'] = list(jason['sequence'])
    jason['structure'] = list(jason['structure'])
    jason['predicted_loop_type'] = list(jason['predicted_loop_type'])


In [4]:
for jason in data_test:
    jason['step'] = list(range(jason['seq_scored']))
    jason['sequence'] = list(jason['sequence'])
    jason['structure'] = list(jason['structure'])
    jason['predicted_loop_type'] = list(jason['predicted_loop_type'])


In [5]:
train = pd.json_normalize(data = data_train, 
                            record_path ='reactivity',  
                            meta =['id','signal_to_noise',
                                  'SN_filter','seq_length','seq_scored']) 
train.rename(columns={0:'reactivity'}, inplace=True)
train['step'] = pd.json_normalize(data = data_train, 
                            record_path ='step'
                                        )
train['sequence'] = pd.json_normalize(data = data_train, 
                            record_path ='sequence'
                                        )
train['structure'] = pd.json_normalize(data = data_train, 
                            record_path ='structure'
                                        )
train['predicted_loop_type'] = pd.json_normalize(data = data_train, 
                            record_path ='predicted_loop_type'
                                        )
train['reactivity_error'] = pd.json_normalize(data = data_train, 
                            record_path ='reactivity_error'
                                        )
train['deg_Mg_pH10'] = pd.json_normalize(data = data_train, 
                            record_path ='deg_Mg_pH10'
                                        )
train['deg_error_Mg_pH10'] = pd.json_normalize(data = data_train, 
                            record_path ='deg_error_Mg_pH10'
                                        )
train['deg_pH10'] = pd.json_normalize(data = data_train, 
                            record_path ='deg_pH10',
                                        )
train['deg_error_pH10'] = pd.json_normalize(data = data_train, 
                            record_path ='deg_error_pH10',
                                        )
train['deg_Mg_50C'] = pd.json_normalize(data = data_train, 
                            record_path ='deg_Mg_50C',
                                        )
train['deg_error_Mg_50C'] = pd.json_normalize(data = data_train, 
                            record_path ='deg_error_Mg_50C',
                                        )
train['deg_50C'] = pd.json_normalize(data = data_train, 
                            record_path ='deg_50C',
                                        )
train['deg_error_50C'] = pd.json_normalize(data = data_train, 
                            record_path ='deg_error_50C',
                                        )

train.set_index(['id','step'], inplace=True)


In [6]:
test = pd.json_normalize(data = data_test,
                         record_path = 'sequence',
                        meta = ['id','seq_length','seq_scored'])
test.rename(columns={0:'sequence'},inplace=True)
test['step'] = pd.json_normalize(data = data_test,
                                record_path = 'step')
test['sequence'] = pd.json_normalize(data = data_test,
                                    record_path = 'sequence')
test['structure'] = pd.json_normalize(data = data_test,
                                     record_path = 'structure')
test['predicted_loop_type'] = pd.json_normalize(data = data_test,
                                               record_path = 'predicted_loop_type')
test.set_index(['id','step'], inplace=True)


In [7]:
train


reactivity signal_to_noise SN_filter seq_length seq_scored  \
id           step                                                               
id_001f94081 0         0.3297           6.894         1        107         68   
             1         1.5693           6.894         1        107         68   
             2         1.1227           6.894         1        107         68   
             3         0.8686           6.894         1        107         68   
             4         0.7217           6.894         1        107         68   
...                       ...             ...       ...        ...        ...   
id_fff546103 63        0.0272           6.545         1        107         68   
             64        0.0382           6.545         1        107         68   
             65        0.0381           6.545         1        107         68   
             66       -0.0066           6.545         1        107         68   
             67        0.0706           6.545         1        107         68   

                  sequence structure predicted_loop_type  reactivity_error  \
id           step                                                            
id_001f94081 0           G         .                   E            0.1359   
             1           G         .                   E            0.2070   
             2           A         .                   E            0.1633   
             3           A         .                   E            0.1452   
             4           A         .                   E            0.1314   
...                    ...       ...                 ...               ...   
id_fff546103 63          A         (                   S            0.0267   
             64          A         (                   S            0.0219   
             65          G         (                   S            0.0218   
             66          G         (                   S            0.0220   
             67          U         .                   H            0.0287   

                   deg_Mg_pH10  deg_error_Mg_pH10  deg_pH10  deg_error_pH10  \
id           step                                                             
id_001f94081 0          0.7556             0.2613    2.3375          0.2631   
             1          2.9830             0.3842    3.5060          0.2860   
             2          0.2526             0.1372    0.3008          0.0964   
             3          1.3789             0.2531    1.0108          0.1574   
             4          0.6376             0.1798    0.2635          0.1000   
...                        ...                ...       ...             ...   
id_fff546103 63         0.5059             0.0626    0.4998          0.0996   
             64         0.2025             0.0411    0.3057          0.0760   
             65         0.1827             0.0391    0.1642          0.0611   
             66         0.3261             0.0509    0.1654          0.0719   
             67         0.4529             0.0565    0.4840          0.0936   

                   deg_Mg_50C  deg_error_Mg_50C  deg_50C  deg_error_50C  
id           step                                                        
id_001f94081 0         0.3581            0.1501   0.6382         0.2167  
             1         2.9683            0.2750   3.4773         0.3475  
             2         0.2589            0.0947   0.9988         0.1880  
             3         1.4552            0.1866   1.3228         0.2124  
             4         0.7244            0.1369   0.7877         0.1703  
...                       ...               ...      ...            ...  
id_fff546103 63        0.2870            0.0638   0.1820         0.0809  
             64        0.2022            0.0520   0.2038         0.0748  
             65        0.2047            0.0520   0.1120         0.0612  
             66        0.2447            0.0595   0.1186         0.0723  
             67        0.1802     

In [8]:
test


sequence seq_length seq_scored structure predicted_loop_type
id           step                                                             
id_00b436dec 0.0         G        107         68         .                   E
             1.0         G        107         68         .                   E
             2.0         A        107         68         .                   E
             3.0         A        107         68         .                   E
             4.0         A        107         68         .                   E
...                    ...        ...        ...       ...                 ...
id_ff103a85c NaN         A        107         68         .                   E
             NaN         C        107         68         .                   E
             NaN         A        107         68         .                   E
             NaN         A        107         68         .                   E
             NaN         C        107         68         .                   E

[25680 rows x 5 columns]

In [9]:
np.random.seed(2020) #Seed the randomness to be deterministic


In [10]:
enc = LabelEncoder()
category_cols_train = [cols for cols in train.columns if train[cols].dtype == 'object']
category_cols_test = [cols for cols in test.columns if test[cols].dtype == 'object']
print(category_cols_train)
print(category_cols_test)
X_train_enc = deepcopy(train)
X_test_enc = deepcopy(test)
for cols in category_cols_train:
    X_train_enc[cols] = enc.fit_transform(X_train_enc[cols])
for cols in category_cols_test:
    X_test_enc[cols] = enc.fit_transform(X_test_enc[cols])


['signal_to_noise', 'SN_filter', 'seq_length', 'seq_scored', 'sequence', 'structure', 'predicted_loop_type']
['sequence', 'seq_length', 'seq_scored', 'structure', 'predicted_loop_type']


In [11]:
X = X_train_enc.drop(['reactivity', 'deg_Mg_pH10', 'deg_pH10', 'deg_Mg_50C', 'deg_50C'],axis=1)
y = X_train_enc.loc[:,['reactivity', 'deg_Mg_pH10', 'deg_pH10', 'deg_Mg_50C', 'deg_50C']]


In [12]:
hgbr = MultiOutputRegressor(HistGradientBoostingRegressor(max_iter=1000,early_stopping=True,n_iter_no_change=10,
                                     learning_rate=0.0025,tol=1e-5,verbose=2)
                           ,n_jobs = 4)


In [13]:
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2)


In [14]:
X_train


,,signal_to_noise,SN_filter,seq_length,seq_scored,sequence,structure,predicted_loop_type,reactivity_error,deg_error_Mg_pH10,deg_error_pH10,deg_error_Mg_50C,deg_error_50C
id,step,,,,,,,,,,,,
id_6cb65a015,15,1562,0,0,0,0,2,2,0.0703,0.0547,0.0898,0.0490,0.0979
id_4d9e01b6c,51,968,1,0,0,2,2,2,0.0269,0.0504,0.0757,0.0366,0.0552
id_b726b998a,59,1561,1,0,0,1,0,5,0.0322,0.0324,0.0443,0.0255,0.0454
id_6add5e119,55,1140,1,0,0,1,1,5,0.0470,0.0582,0.1067,0.0645,0.1071
id_f32da8012,50,216,0,0,0,1,2,1,0.1142,0.1826,0.2554,0.1391,0.2900
...,...,...,...,...,...,...,...,...,...,...,...,...,...
id_a875d91d3,32,841,1,0,0,0,2,1,0.0864,0.1232,0.1559,0.0974,0.1783
id_219d8c69a,47,1660,1,0,0,0,2,2,0.0236,0.0207,0.0419,0.0267,0.0431
id_1f465a329,27,1505,1,0,0,0,2,2,0.0365,0.1212,0.1199,0.0986,0.0989


In [15]:
hgbr.fit(X_train,y_train)


Binning 0.010 GB of training data: Binning 0.010 GB of training data: Binning 0.010 GB of training data: Binning 0.010 GB of training data: 0.132 s
Binning 0.001 GB of validation data: 0.134 s
Binning 0.001 GB of validation data: 0.005 s
Fitting gradient boosted rounds:
[1/1000] 0.005 s
Fitting gradient boosted rounds:
[1/1000] 0.125 s
Binning 0.001 GB of validation data: 0.005 s
Fitting gradient boosted rounds:
[1/1000] 1 tree, 31 leaves, max depth = 14, train loss: 0.86483, val loss: 0.84308, in 0.010s
[2/1000] 1 tree, 31 leaves, max depth = 10, train loss: 0.28187, val loss: 0.29610, in 0.011s
[2/1000] 1 tree, 31 leaves, max depth = 10, train loss: 0.40364, val loss: 0.29568, in 0.011s
[2/1000] 1 tree, 31 leaves, max depth = 11, train loss: 0.86436, val loss: 0.84265, in 0.010s
[3/1000] 1 tree, 31 leaves, max depth = 10, train loss: 0.28158, val loss: 0.29583, in 0.011s
[3/1000] 1 tree, 31 leaves, max depth = 14, train loss: 0.86388, val loss: 0.84225, in 0.009s
[4/1000] 1 tree, 31 

MultiOutputRegressor(estimator=HistGradientBoostingRegressor(early_stopping=True,
                                                             learning_rate=0.0025,
                                                             max_iter=1000,
                                                             tol=1e-05,
                                                             verbose=2),
                     n_jobs=4)

In [16]:
y_pred = hgbr.predict(X_test)


In [17]:
print(mean_squared_error(y_test,y_pred,squared=False))


0.9527754952037544


In [18]:
X_train_enc[X_test_enc.columns]


sequence  seq_length  seq_scored  structure  \
id           step                                                
id_001f94081 0            2           0           0          2   
             1            2           0           0          2   
             2            0           0           0          2   
             3            0           0           0          2   
             4            0           0           0          2   
...                     ...         ...         ...        ...   
id_fff546103 63           0           0           0          0   
             64           0           0           0          0   
             65           2           0           0          0   
             66           2           0           0          0   
             67           3           0           0          2   

                   predicted_loop_type  
id           step                       
id_001f94081 0                       1  
             1                       1  
             2                       1  
             3                       1  
             4                       1  
...                                ...  
id_fff546103 63                      5  
             64                      5  
             65                      5  
             66                      5  
             67                      2  

[146880 rows x 5 columns]

In [19]:
hgbr.fit(X_train_enc[X_test_enc.columns],y)


1 tree, 31 leaves, max depth = 17, train loss: 0.30569, val loss: 0.21854, in 0.010s
Fit 1000 trees in 10.544 s, (31000 total leaves)
Time spent computing histograms: 5.519s
Time spent finding best splits:  0.471s
Time spent applying splits:      2.599s
Time spent predicting:           0.215s
Binning 0.005 GB of training data: 1 tree, 31 leaves, max depth = 12, train loss: 0.20791, val loss: 0.22916, in 0.010s
Fit 1000 trees in 11.958 s, (31000 total leaves)
Time spent computing histograms: 6.169s
Time spent finding best splits:  0.575s
Time spent applying splits:      3.021s
Time spent predicting:           0.235s
Binning 0.005 GB of training data: 1 tree, 31 leaves, max depth = 12, train loss: 0.17990, val loss: 0.20408, in 0.009s
Fit 1000 trees in 10.642 s, (31000 total leaves)
Time spent computing histograms: 5.558s
Time spent finding best splits:  0.492s
Time spent applying splits:      2.656s
Time spent predicting:           0.217s
Binning 0.005 GB of training data: 1 tree, 31 le

MultiOutputRegressor(estimator=HistGradientBoostingRegressor(early_stopping=True,
                                                             learning_rate=0.0025,
                                                             max_iter=1000,
                                                             tol=1e-05,
                                                             verbose=2),
                     n_jobs=4)

In [20]:
y_pred_2 = hgbr.predict(X_test_enc)


In [21]:
y_pred_2.shape


(25680, 5)

In [22]:
submission = pd.DataFrame(np.concatenate([test_set.id_seqpos.values[:,np.newaxis],y_pred_2],axis=1),columns=test_set.columns)
submission.head(10)
submission.to_csv('submission.csv',index=False)
